# 04 Current snapshot


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes requests papermill
import os,sys,json,pandas as pd
from google.cloud import bigquery
PROJECT_ID="your-gcp-project-id"  # замініть у всіх ноутбуках
LOCATION="EU"; creds=None
if "google.colab" in sys.modules:
 from google.colab import auth; auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
 from google.oauth2 import service_account
 creds=service_account.Credentials.from_service_account_info(json.loads(os.environ["GCP_SA_KEY"]),scopes=["https://www.googleapis.com/auth/cloud-platform"])
client=bigquery.Client(project=PROJECT_ID,location=LOCATION,credentials=creds)
print("проєкт:",client.project)

In [ ]:
R=f"{PROJECT_ID}.nbu_raw.raw_rates"; D=f"{PROJECT_ID}.nbu_dwh.dim_currency"; T=f"{PROJECT_ID}.nbu_dwh.snapshot_rates_current"; r=client.query(f"SELECT ingested_at,business_date,payload FROM `{R}`").to_dataframe(); p=pd.json_normalize(r.payload.map(json.loads)); d=pd.concat([r[["ingested_at","business_date"]].reset_index(drop=True),p],axis=1); d["currency_code"]=d.cc.astype(str).str.strip().str.upper(); d["currency_name"]=d.txt.astype(str).str.strip(); d["rate"]=pd.to_numeric(d.rate); o=d.sort_values(["business_date","ingested_at"]).drop_duplicates("currency_code",keep="last")[["currency_code","currency_name","rate","business_date"]].merge(client.query(f"SELECT currency_key,currency_code FROM `{D}`").to_dataframe(),on="currency_code",how="left"); o.currency_key=o.currency_key.fillna(-1).astype(int); o["snapshot_ts"]=pd.Timestamp.now(tz="UTC"); o=o[["currency_key","currency_code","currency_name","rate","business_date","snapshot_ts"]]
s=[bigquery.SchemaField("currency_key","INTEGER"),bigquery.SchemaField("currency_code","STRING"),bigquery.SchemaField("currency_name","STRING"),bigquery.SchemaField("rate","FLOAT"),bigquery.SchemaField("business_date","DATE"),bigquery.SchemaField("snapshot_ts","TIMESTAMP")]; client.load_table_from_dataframe(o,T,job_config=bigquery.LoadJobConfig(schema=s,write_disposition="WRITE_TRUNCATE")).result(); assert len(o)==d.currency_code.nunique() and o.currency_code.is_unique and not (o.currency_key==-1).any(); display(o.head()); print("checks: OK")